# Cryptography Tutorial Using The `cryptography` Library
This notebook demonstrates how to manage virtual environments and use the explicit `cryptography.hazmat` package for complete cryptographic operations.


## 1. Base64 Encoding and Decoding
Base64 built-in library.


In [ ]:
import base64

message = b'Secret message that needs to be encoded'
encoded = base64.b64encode(message)
print('Encoded:', encoded)

decoded = base64.b64decode(encoded)
print('Decoded:', decoded)


## 2. Hashing
`hashlib` is the standard library which uses OpenSSL's fast hashing implementations under the hood.


In [ ]:
import hashlib

data = b'This is some data to hash'
algo = 'sha256'  # Can change to sha512, sha3_256, etc.
hash_obj = hashlib.new(algo)
hash_obj.update(data)
print(f'{algo} Hash:', hash_obj.hexdigest())


## 3. Symmetric Encryption (Explicit Algorithms)
Here we use explicit algorithms (like AES) and modes of operation (like CBC). We will write the encrypted ciphertext directly to a file as hex, then read it back to decrypt it.


In [ ]:
import os
import binascii
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import padding

# -- Configuration --
# For an AES-256 key, we need 32 bytes of randomness.
key = os.urandom(32)
# Initialization Vector (IV) for CBC mode needs to be block_size / 8 bytes. For AES this is 16 bytes.
iv = os.urandom(16)

# Explicitly selecting AES as the algorithm and CBC as the mode
cipher_algo = algorithms.AES(key)
cipher_mode = modes.CBC(iv)

cipher = Cipher(cipher_algo, cipher_mode, backend=default_backend())

# -- Encryption --
message = b"This is highly classified symmetric data that we need to protect."
print(f"Original plaintext: {message.decode('utf-8')}")

# In Block Ciphers like CBC, the message MUST be a multiple of the block size (128 bits for AES).
# We use PKCS7 padding to achieve this automatically.
padder = padding.PKCS7(algorithms.AES.block_size).padder()
padded_data = padder.update(message) + padder.finalize()

encryptor = cipher.encryptor()
ciphertext = encryptor.update(padded_data) + encryptor.finalize()

# Save the encrypted ciphertext to a file as hex string 
hex_ciphertext = binascii.hexlify(ciphertext).decode('utf-8')
with open("symmetric_encrypted.txt", "w") as f:
    f.write(hex_ciphertext)

print(f"Encrypted and saved to symmetric_encrypted.txt as HEX: {hex_ciphertext[:64]}...")


In [ ]:
# -- Decryption --
# Read ciphertext from file
with open("symmetric_encrypted.txt", "r") as f:
    read_hex = f.read()

# Convert back from HEX to bytes
read_ciphertext = binascii.unhexlify(read_hex)

# Decrypt using the same key, IV, algorithm, and mode!
decryptor = cipher.decryptor()
padded_decrypted = decryptor.update(read_ciphertext) + decryptor.finalize()

# Remove the PKCS7 padding
unpadder = padding.PKCS7(algorithms.AES.block_size).unpadder()
decrypted_message = unpadder.update(padded_decrypted) + unpadder.finalize()

print(f"Decrypted plaintext: {decrypted_message.decode('utf-8')}")


## 4. Asymmetric Operations (Explicit Algorithms)
Asymmetric encryption uses a key pair: a private key and a public key. We will write the encrypted data out as a hex file just like before.

### Step 4.1: Key Pair Generation


In [19]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

# Explicitly choosing RSA for Key Generation
print("Generating RSA Keypair...")
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
    backend=default_backend()
)
public_key = private_key.public_key()

print("Keys generated successfully.")


Generating RSA Keypair...
Keys generated successfully.


### Step 4.2: Asymmetric Encryption


In [ ]:
from cryptography.hazmat.primitives.asymmetric import padding as asym_padding
from cryptography.hazmat.primitives import hashes

asym_message = b"This secret is protected via Public Key infrastructure."
print(f"Message to encrypt: {asym_message}")

# Explicitly choosing PKCS1v15 padding algorithm (default for OpenSSL)
chosen_padding = asym_padding.PKCS1v15()

encrypted = public_key.encrypt(
    asym_message,
    chosen_padding
)

# Save encrypted to file as hex
hex_asym = binascii.hexlify(encrypted).decode('utf-8')
with open("asymmetric_encrypted.txt", "w") as f:
    f.write(hex_asym)

print(f"Encrypted and saved to asymmetric_encrypted.txt as HEX: {hex_asym[:64]}...")


### Step 4.3: Asymmetric Decryption


In [ ]:
# Read ciphertext from file
with open("asymmetric_encrypted.txt", "r") as f:
    read_asym_hex = f.read()

read_asym_encrypted = binascii.unhexlify(read_asym_hex)

# Decrypt using the Private Key and the EXACT same padding and hashing algorithms
decrypted = private_key.decrypt(
    read_asym_encrypted,
    chosen_padding
)

print(f"Decrypted plaintext: {decrypted.decode('utf-8')}")


### Step 4.4: Cryptographic Signing
Signing proves who created a file and that it hasn't been altered.


In [ ]:
document_to_sign = b"This is a legally binding contract."

# Explicitly choosing PKCS1v15 padding and SHA256 for the signature
sign_padding = asym_padding.PKCS1v15()
sign_hash_algo = hashes.SHA256()

print(f"Signing document using {sign_hash_algo.name}...")
signature = private_key.sign(
    document_to_sign,
    sign_padding,
    sign_hash_algo
)
print(f"Signature created as HEX: {binascii.hexlify(signature).decode('utf-8')[:64]}...")


### Step 4.5: Signature Verification
Anyone with our **Public Key** can verify the signature.


In [ ]:
print("Verifying signature with Public Key...")
try:
    public_key.verify(
        signature,
        document_to_sign,
        sign_padding,
        sign_hash_algo
    )
    print("SUCCESS: The signature is valid and authentic.")
except Exception as e:
    print("FAILURE: The signature is invalid.", e)
